In [ ]:
ver = "v3"

# Mapping of descriptive set names to list of associated file names
set_definitions = {
    "GAR_full": ["GAR_subset_full"],
    "GAR_LLPS_pos": [
        "4_LLPS_positive_set_and_GAR_subset",
        "5_LLPS_positive_set_and_NA_positive_set_and_GAR_subset"
    ],
    "GAR_LLPS_pos_NA_neg": [
        "4_LLPS_positive_set_and_GAR_subset"
    ],
    "GAR_LLPS_neg": [
        "6_NA_positive_set_and_GAR_subset",
        "7_GAR_subset_only"
    ],
    "GAR_LLPS_neg_NA_pos": [
        "6_NA_positive_set_and_GAR_subset"
    ],
    "GAR_NA_pos": [
        "5_LLPS_positive_set_and_NA_positive_set_and_GAR_subset",
        "6_NA_positive_set_and_GAR_subset"
    ],
    "GAR_NA_neg": [
        "4_LLPS_positive_set_and_GAR_subset",
        "7_GAR_subset_only"
    ],
    "GAR_pos": [
        "5_LLPS_positive_set_and_NA_positive_set_and_GAR_subset"
    ],
    "GAR_neg": [
        "7_GAR_subset_only"
    ],
}

# Initialize containers
set_list = []
set_dict = {}
proteins_sets_dict = {}

# Read each set from its associated file(s)
for set_name, file_names in set_definitions.items():
    proteins = []
    for fname in file_names:
        file_path = f"/mnt/d/phd/scripts/11_pub_hu-RG-motif-composition_analysis/data//processed/final_set_lists/{fname}.txt"
        with open(file_path, "r") as fl:
            proteins.extend([line.strip() for line in fl])
    set_dict[set_name] = proteins
    set_list.append(proteins)

# Add full proteome
full_proteome_path = f"/mnt/d/phd/scripts/11_pub_hu-RG-motif-composition_analysis/data/processed/list_of_human_proteins.csv"
with open(full_proteome_path, "r") as fl:
    full_proteome = [line.strip() for line in fl]
set_dict["full_proteome"] = full_proteome
set_list.append(full_proteome)

# Store all in versioned dictionary
set_names = list(set_dict.keys())
proteins_sets_dict[ver] = set_dict


In [ ]:
def assign_groups_advanced(protein, group1_name: list, group1_string, group2_name: list, group2_string, group3_name: list, group3_string, group4_name: list, group4_string ):
    if protein in group1_name:
        return group1_string
    elif protein in group2_name:
        return group2_string
    elif protein in group3_name:
        return group3_string
    elif protein in group4_name:
        return group4_string
    else:
        return 'Not in any group'

# Standard libraries
import os

# Third-party libraries
import pandas as pd

curr_wd = os.path.abspath(os.getcwd())
print(f"Current working directory: {curr_wd}")


file_path = os.path.abspath(os.path.join(curr_wd, '..', '11_pub_hu-RG-motif-composition_analysis/data/processed/GAR_motif_Wang_set_human_cleaned_annot_filtered.parquet'))
print(file_path)

motif_info_set_df = pd.read_parquet(file_path)

motif_info_set_df['Groups_num'] = motif_info_set_df['UniqueID'].apply(
    assign_groups_advanced,
    args=(
        set_list[set_names.index("GAR_pos")], "5",
        set_list[set_names.index("GAR_neg")], "7",
        set_list[set_names.index("GAR_LLPS_pos_NA_neg")], "4",
        set_list[set_names.index("GAR_LLPS_neg_NA_pos")], "6"
    )
)

motif_info_set_df['start_ext_30'] = (motif_info_set_df['start'] - 30).clip(lower=0)
motif_info_set_df['end_ext_30'] = motif_info_set_df.apply(
    lambda row: min(row['end'] + 30, len(row['full_seq'])), axis=1
)
motif_info_set_df['seq_ext_30'] = motif_info_set_df.apply(
    lambda row: row['full_seq'][row['start_ext_30']:row['end_ext_30']],
    axis=1
)

motif_info_set_df['start_ext_40'] = (motif_info_set_df['start'] - 40).clip(lower=0)
motif_info_set_df['end_ext_40'] = motif_info_set_df.apply(
    lambda row: min(row['end'] + 40, len(row['full_seq'])), axis=1
)
motif_info_set_df['seq_ext_40'] = motif_info_set_df.apply(
    lambda row: row['full_seq'][row['start_ext_40']:row['end_ext_40']],
    axis=1
)
#### add the region of 30 amino acids left and right

In [ ]:
import pandas as pd

# Assuming your dataframe is called df
# df = pd.read_csv("your_file.csv")  # if needed

for i in range(4,8):
    print(i)

    fasta_path = f"RGmotifs_extended40_group_{i}.fasta"

    with open(fasta_path, "w") as f:
        for _, row in motif_info_set_df[motif_info_set_df["Groups_num"] == str(i)].iterrows():
            unique_id = row["UniqueID"]
            start = row["start_ext_40"]
            end = row["end_ext_40"]
            seq = row["seq_ext_40"]

            # Build header: UniqueID_start-end
            header = f">{unique_id}_{start}-{end}"

            # Write to FASTA
            f.write(f"{header}\n{seq}\n")

In [ ]:
################ EXTRACTION ######################

In [ ]:
import requests
from Bio.Seq import Seq

# ================================================================
#   Helper: Parse proteinLocation safely for all UniProt cases
# ================================================================
def parse_protein_location(pl):
    """
    Handles all forms:
    - {"position": 57}
    - {"position": {"position": 57}}
    - {"begin": {"position": 12}, "end": {"position": 35}}
    """
    if "position" in pl:
        pos = pl["position"]
        if isinstance(pos, dict):
            return int(pos["position"]), int(pos["position"])
        else:
            return int(pos), int(pos)

    elif "begin" in pl and "end" in pl:
        return int(pl["begin"]["position"]), int(pl["end"]["position"])

    else:
        raise ValueError(f"Unexpected proteinLocation structure: {pl}")


# ================================================================
#   Fetch sequence from Ensembl
# ================================================================
def fetch_seq(chrom, start, end, species="human"):
    url = f"https://rest.ensembl.org/sequence/region/{species}/{chrom}:{start}..{end}:1"
    r = requests.get(url, headers={"Content-Type": "text/plain"})
    r.raise_for_status()
    return r.text.strip()


# ================================================================
#   FORWARD STRAND ONLY
# ================================================================
def get_exact_coding_dna_forward_only(
    entry,
    aa_start,
    aa_end,
    protein_id,
    species="human",
    prot_seq=None
):

    # -----------------------------
    # Parse exons
    # -----------------------------
    exons = []
    for ex in entry["genomicLocation"]["exon"]:
        p_start, p_end = parse_protein_location(ex["proteinLocation"])
        # print(p_start)
        # print(p_end)
        # print(ex)
        if "begin" not in ex["genomeLocation"]:
            g1,g2 = parse_protein_location(ex["genomeLocation"])
        else:
            g1 = ex["genomeLocation"]["begin"]["position"]
            g2 = ex["genomeLocation"]["end"]["position"]
        g_start = int(min(g1, g2))
        g_end = int(max(g1, g2))

        exons.append({
            "p_start": p_start,
            "p_end": p_end,
            "g_start": g_start,
            "g_end": g_end,
            "len": g_end - g_start + 1
        })

    # Sort by protein location
    exons.sort(key=lambda e: e["p_start"])

    first_aa = exons[0]["p_start"]
    last_aa = exons[-1]["p_end"]

    # -----------------------------
    # Fetch all exon sequences
    # -----------------------------
    exon_seqs = []
    exon_cum = []
    cum = 0

    for ex in exons:
        seq = fetch_seq(entry["genomicLocation"]["chromosome"], ex["g_start"], ex["g_end"], species)
        if len(seq) != ex["len"]:
            print("Warning: exon length mismatch")
        exon_seqs.append(seq)
        exon_cum.append(cum)
        cum += len(seq)

    concatenated = "".join(exon_seqs)
    total_nt = len(concatenated)

    # -----------------------------
    # Determine global nt coords
    # -----------------------------
    global_nt_start = (aa_start - first_aa) * 3
    global_nt_end   = (aa_end   - first_aa) * 3 + 2

    global_nt_start = max(0, global_nt_start)
    global_nt_end = min(total_nt - 1, global_nt_end)

    # -----------------------------
    # Build genomic intervals
    # -----------------------------
    intervals = []
    for ex, cum_start in zip(exons, exon_cum):
        exon_len = ex["len"]

        ov_s = max(global_nt_start, cum_start)
        ov_e = min(global_nt_end, cum_start + exon_len - 1)
        if ov_s > ov_e:
            continue

        local_s = ov_s - cum_start
        local_e = ov_e - cum_start

        iv_start = ex["g_start"] + local_s
        iv_end   = ex["g_start"] + local_e

        intervals.append({
            "chrom": entry["genomicLocation"]["chromosome"],
            "start": int(iv_start),
            "end":   int(iv_end),
            "strand": "+"
        })

    # Merge adjacent intervals
    merged = []
    for iv in intervals:
        if not merged:
            merged.append(iv)
        else:
            prev = merged[-1]
            if iv["chrom"] == prev["chrom"] and iv["start"] <= prev["end"] + 1:
                prev["end"] = max(prev["end"], iv["end"])
            else:
                merged.append(iv)

    # -----------------------------
    # DNA sequence in correct orientation
    # -----------------------------
    dna_seq = concatenated[global_nt_start: global_nt_end + 1]

    # Translation check
    if prot_seq is not None:
        trans = str(Seq(dna_seq).translate())
        expected = prot_seq[aa_start - 1: aa_end]
        if trans != expected:
            print("⚠⚠⚠ For ", protein_id,  aa_start , "-", aa_end, " there is a translation mismatch! ⚠⚠⚠")
            print("Translated:", trans)
            print("Acquired DNA sequence:", dna_seq)
            print("UniProt: ",expected )
            print("Genomic coordinates:", intervals) 
            print("________________")
        else: 
            print("For ", protein_id , " the translation fits given sequence!")
    else:
        trans = None

    return intervals, dna_seq, trans



# ================================================================
#   REVERSE STRAND ONLY
#   (Your function, now cleaned + unified proteinLocation handler)
# ================================================================
def get_exact_coding_dna_reverse_only(
    entry,
    aa_start,
    aa_end,
    protein_id,
    species="human",
    prot_seq=None
):
    chrom = entry["genomicLocation"]["chromosome"]
    # -----------------------------
    # Parse exons
    # -----------------------------
    exons = []
    for ex in entry["genomicLocation"]["exon"]:
        p_start, p_end = parse_protein_location(ex["proteinLocation"])
        if "begin" not in ex["genomeLocation"]:
            g1,g2 = parse_protein_location(ex["genomeLocation"])
        else:
            g1 = ex["genomeLocation"]["begin"]["position"]
            g2 = ex["genomeLocation"]["end"]["position"]

        # g1 = ex["genomeLocation"]["begin"]["position"]
        # g2 = ex["genomeLocation"]["end"]["position"]
        g_start = int(min(g1, g2))
        g_end = int(max(g1, g2))

        exons.append({
            "p_start": p_start,
            "p_end": p_end,
            "g_start": g_start,
            "g_end": g_end,
            "len": g_end - g_start + 1
        })

    # Protein-order sort
    exons.sort(key=lambda e: e["p_start"])
    # print(exons)
    first_aa = exons[0]["p_start"]
    last_aa  = exons[-1]["p_end"]

    # -----------------------------
    # Fetch exon sequences and RC
    # -----------------------------
    exon_seqs_rc = []
    exon_cum = []
    cum = 0

    for ex in exons:
        # print(ex)
        seq = fetch_seq(chrom, ex["g_start"], ex["g_end"], species)
        if len(seq) != ex["len"]:
            print("Warning: exon length mismatch")
        rc = str(Seq(seq).reverse_complement())
        exon_seqs_rc.append(rc)
        exon_cum.append(cum)
        cum += len(rc)
    # print(exon_cum)
    concatenated = "".join(exon_seqs_rc)
    total_nt = len(concatenated)
    # print('concat: ', concatenated)
    # -----------------------------
    # AA → nt coordinates in RC transcript
    # -----------------------------
    global_nt_start = (aa_start - first_aa) * 3
    global_nt_end = (aa_end - first_aa) * 3 + 2

    global_nt_start = max(0, global_nt_start)
    global_nt_end = min(total_nt - 1, global_nt_end)

    # -----------------------------
    # Build genomic intervals
    # -----------------------------
    intervals = []
    for ex, cum_start, rc_seq in zip(exons, exon_cum, exon_seqs_rc):
        exon_len = ex["len"]

        ov_s = max(global_nt_start, cum_start)
        ov_e = min(global_nt_end, cum_start + exon_len - 1)
        if ov_s > ov_e:
            continue

        local_s = ov_s - cum_start
        local_e = ov_e - cum_start

        # Reverse mapping: local index j → genomic position = g_start + (exon_len - 1 - j)
        g_s = ex["g_start"] + (exon_len - 1 - local_s)
        g_e = ex["g_start"] + (exon_len - 1 - local_e)

        iv_start = min(g_s, g_e)
        iv_end   = max(g_s, g_e)

        intervals.append({"chrom": chrom, "start": iv_start, "end": iv_end, "strand": "-"})
    # print(intervals)
    # Merge
    # merged = []
    # for iv in intervals:
    #     if not merged:
    #         merged.append(iv)
    #     else:
    #         prev = merged[-1]
    #         if iv["chrom"] == prev["chrom"] and iv["start"] <= prev["end"] + 1:
    #             prev["end"] = max(prev["end"], iv["end"])
    #         else:
    #             merged.append(iv)
    # print('merged:', merged)
    # Extract DNA
    dna_seq = concatenated[global_nt_start: global_nt_end + 1]

    # Translation check
    if prot_seq is not None:
        trans = str(Seq(dna_seq).translate())
        expected = prot_seq[aa_start - 1: aa_end]
        if trans != expected:
            print("⚠⚠⚠ For ", protein_id,  aa_start , "-", aa_end, " there is a translation mismatch! ⚠⚠⚠")
            print("Translated:", trans)
            print("Acquired DNA sequence:", dna_seq)
            print("UniProt: ",expected )
            print("Genomic coordinates:", intervals) 
            print("________________")
        else: 
            print("For ", protein_id , " the translation fits given sequence!")
    else:
        trans = None

    return intervals, dna_seq, trans



# ================================================================
#   UNIFIED FUNCTION: works for any transcript
# ================================================================
def get_exact_coding_dna(
    protein_id,
    start_residue,
    end_residue,
    flank_residues=0,
    protein_length=None,
    species="human",
    validate_translation=True
):
    """
    Automatically detects forward or reverse strand
    and extracts the exact coding DNA.
    """

    # --------------------
    # Download UniProt/EBI entry
    # --------------------
    url = f"https://www.ebi.ac.uk/proteins/api/coordinates/{protein_id}"
    r = requests.get(url, headers={"Accept": "application/json"})
    # print(r)
    if r.status_code == 404:
        print("No genomic information in UniProt for:", protein_id)
        return None, None, None
    r.raise_for_status()
    data = r.json()[0] if isinstance(r.json(), list) else r.json()

    if "gnCoordinate" not in data:
        raise ValueError("No genomic coordinates available")

    entry = data["gnCoordinate"][0]
    prot_seq = data.get("sequence", None)

    # --------------------
    # Apply flanking
    # --------------------
    if flank_residues > 0 and protein_length is None:
        raise ValueError("protein_length required when using flanks")

    aa_start = max(1, start_residue - flank_residues)
    aa_end = end_residue + flank_residues if protein_length else end_residue

    if protein_length is not None:
        aa_end = min(aa_end, protein_length)

    # --------------------
    # Detect strand
    # --------------------
    reverse = entry["genomicLocation"]["reverseStrand"]
    # print(reverse)
    if not reverse:
        intervals, dna, trans = get_exact_coding_dna_forward_only(
            entry, aa_start, aa_end, protein_id, species, prot_seq if validate_translation else None
        )
    else:
        intervals, dna, trans = get_exact_coding_dna_reverse_only(
            entry, aa_start, aa_end, protein_id, species, prot_seq if validate_translation else None
        )
    # print('intervals: ', intervals)
    return intervals, dna, trans



# ================================================================
#   PROCESS MANY REGIONS
# ================================================================
def process_multiple_regions(region_list):
    """
    region_list = [
        {
            "protein": "Q9Y4Z0",
            "start": 1,
            "end": 10,
            "flank": 0,
            "protein_length": 139
        },
        ...
    ]
    """
    results = []
    for r in region_list:
        print(r[0])
        iv, dna, trans = get_exact_coding_dna(
            # r["protein"],
            # r["start"],
            # r["end"],
            # flank_residues=r.get("flank", 0),
            # protein_length=r.get("protein_length", None)
            r[0],
            r[1],
            r[2]
        )
        if iv == None:
            continue
        results.append({"protein": r[0],"prot_region": (r[1], r[2]),"prot_seq": trans, "intervals": iv, "dna": dna})
    return results


def write_bed_from_results(results, output_file):
    with open(output_file, "w") as f:
        for res in results:
            protein = res["protein"]
            protein_region = res["prot_region"]

            if not res["intervals"]:
                continue

            for iv in res["intervals"]:
                chrom = iv["chrom"]

                # Skip ALT contigs (gnomAD v4 has none of them)
                if chrom.startswith("HSCHR"):
                    continue

                # Add chr prefix if missing
                if not chrom.startswith("chr"):
                    chrom = "chr" + str(chrom)

                start = iv["start"]
                end = iv["end"] + 1
                strand = iv.get("strand", "+")

                name = f"{protein}_{protein_region[0]}_{protein_region[1]}_{chrom}_{start}_{end}_{strand}"

                row = [
                    chrom,
                    str(start),
                    str(end),
                    strand,
                    name,
                    "0",
                    "0",
                    "0",
                    "0",
                    "0",
                    str(start),
                    str(end)
                ]

                f.write("\t".join(row) + "\n")


# def write_bed_from_results(results, output_file):
#     """
#     Write a BED file with 12 columns based on processed regions.
    
#     Args:
#         results (list): Output from process_multiple_regions(), each entry is:
#                         {"protein": str, "intervals": list of dicts, "dna": str}
#         output_file (str): Path to write BED file
#     """
#     with open(output_file, "w") as f:
#         for res in results:
#             protein = res["protein"]
#             protein_region = res["prot_region"]
#             dna_seq = res["dna"]
            
#             if not res["intervals"]:
#                 continue  # skip regions with no intervals
            
#             for iv_idx, iv in enumerate(res["intervals"]):
#                 chrom = iv["chrom"]
#                 start = iv["start"]  # 0-based
#                 end = iv["end"] + 1  # BED end is exclusive
#                 strand = iv.get("strand", "+")
                
#                 # Name: protein + coordinates + strand
#                 name = f"{protein}_{protein_region[0]}_{protein_region[1]}_{chrom}_{start}_{end}_{strand}"
                
#                 # Example motif type (placeholder; you can replace with real)
#                 motif_type = "6mer"
                
#                 # Example transcript ID (placeholder; replace if available)
#                 transcript_id = "ENST00000000000"
                
#                 # Example numeric columns (placeholders)
#                 total_transcript_len = 0
#                 cds_len = 0
#                 orf_len = 0
#                 start_pos = start  # relative position (example)
#                 end_pos = end      # relative position (example)
                
#                 row = [
#                     chrom,
#                     str(start),
#                     str(end),
#                     strand,
#                     name,
#                     motif_type,
#                     transcript_id,
#                     str(total_transcript_len),
#                     str(cds_len),
#                     str(orf_len),
#                     str(start_pos),
#                     str(end_pos)
#                 ]
                
#                 f.write("\t".join(row) + "\n")



# ---------------------------------------------------------
# Test script for unified get_exact_coding_dna()
# ---------------------------------------------------------

# Five diverse proteins:
test_cases = [
    ("Q3KQU3", 325, 333),   # Forward, multi-exon (BRCA1)
    ("Q9Y4Z0", 1, 100),    # Reverse-strand (example you used before)
    # ("P04637", 50, 200),   # TP53 (common use-case)
    # ("Q9H9K5", 5, 250),    # Forward, single-position exon cases
    ("P0CJ87", 1, 140), # Reverse, complex exon boundaries
    # ("P35637", 1, 200),
    # ("P35637", 450, 500),
    # ("Q8TBP5", 67,78)
]

# all_results = []

# for protein_id, aa_start, aa_end in test_cases:
#     print("\n========================================")
#     print(f"Testing {protein_id} | AA {aa_start}-{aa_end}")
#     print("========================================")

#     coords, dna_seq, trans = get_exact_coding_dna(
#         protein_id=protein_id,
#         start_residue=aa_start,
#         end_residue=aa_end,
#         flank_residues=0,
#         protein_length=None,     # automatically fetched
#         validate_translation=True,
#         # verbose=True
#     )

#     print("• Coordinates:", coords)
#     print("• DNA length:", len(dna_seq) if dna_seq else None)
#     print("• DNA seq:", dna_seq)
#     print("• Protein seq:", trans)
#     # print("• Errors so far:", errors)

#     all_results.append((protein_id, coords, dna_seq, trans))


# print("\n========================================")
# print("Finished testing all examples.")
# print("========================================")

# Process regions
results = process_multiple_regions(test_cases)
# print(results)
# Write BED file
write_bed_from_results(results, "test_regions.bed")
print("BED file written to test_regions.bed")



In [ ]:
results

In [ ]:
motif_info_set_df[motif_info_set_df["UniqueID"] == "Q3KQU3"]

In [ ]:
import json

region_list_tup = []
for i,e in motif_info_set_df[motif_info_set_df["Groups_num"] == "7"].iterrows():
    region_list_tup.append((e["UniqueID"], e["start"], e["end"]))


results_neg = process_multiple_regions(region_list_tup)
print(results_neg)
with open('genomic_coordinates_info_neg.json', 'w') as f:
    json.dump(results_neg, f, indent=4)
# Write BED file
write_bed_from_results(results_neg, "neg_RGmotifs_coordinates.bed")
print("BED file written to neg_RGmotifs_coordinates.bed")

In [ ]:
region_list_tup = []
for i,e in motif_info_set_df[motif_info_set_df["Groups_num"] == "5"].iterrows():
    region_list_tup.append((e["UniqueID"], e["start"], e["end"]))


results = process_multiple_regions(region_list_tup)
print(results)
with open('genomic_coordinates_info_pos.json', 'w') as f:
    json.dump(results, f, indent=4)
# Write BED file
write_bed_from_results(results, "pos_RGmotifs_coordinates.bed")
print("BED file written to pos_RGmotifs_coordinates.bed")

In [ ]:
import pandas as pd

group_chosen = "5"

df = motif_info_set_df[motif_info_set_df["Groups_num"] == group_chosen]

cols = ["UniqueID", "start", "end"]

json_data = df[cols].to_json(orient="records", indent=2)
print(json_data)

with open("protein_regions_pos_set_5.json", "w") as f:
    f.write(json_data)